In [0]:
print("Databricks is connected and ready!")

In [0]:
spark.sql("CREATE CATALOG IF NOT EXISTS portfolio")
spark.sql("CREATE SCHEMA IF NOT EXISTS portfolio.raw_data")
spark.sql("CREATE VOLUME IF NOT EXISTS portfolio.raw_data.taxi_files")

In [0]:
import urllib.request

url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
target_path = "/Volumes/portfolio/raw_data/taxi_files/yellow_tripdata_2024-01.parquet"

urllib.request.urlretrieve(url, target_path)
print("Downloaded successfully!")

# Now copy from Volume to S3
dbutils.fs.cp(target_path, "s3://dbx-bronze-silver-gold-portfolio/raw/yellow_tripdata_2024-01.parquet")
print("Copied to S3 successfully!")

In [0]:
from pyspark.sql.functions import current_timestamp, col

df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "parquet")
        .option("cloudFiles.schemaLocation", "s3://dbx-bronze-silver-gold-portfolio/schema/taxi_files")
        .load("s3://dbx-bronze-silver-gold-portfolio/raw/")
)

df_with_meta = (
    df.withColumn("_ingested_at", current_timestamp())
      .withColumn("_source_file", col("_metadata.file_path"))
)

(
    df_with_meta.writeStream
        .format("delta")
        .option("checkpointLocation", "s3://dbx-bronze-silver-gold-portfolio/checkpoints/taxi_files")
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .table("portfolio.raw_data.bronze_taxi_trips")
)

In [0]:
df_bronze = spark.table("portfolio.raw_data.bronze_taxi_trips")
print("Row count:", df_bronze.count())
df_bronze.display()